# 01 – Dataset and Exploratory Data Analysis

**Dataset:** UJIIndoorLoc (UCI ML Repository, CC BY 4.0)

**Goal:** show that the dataset is understood, reliable, and appropriate for predicting
building, floor and coordinates from Wi-Fi fingerprints.

**What one row is:** a single Wi-Fi scan taken by one phone at one location. It records the
signal strength (RSSI) of 520 access points (WAPs), the true location (building, floor,
longitude, latitude), and metadata about who took it, with which phone, and when.

**Rule for this notebook:** we *describe* the validation file but never use its labels to make
modelling decisions. It is our held-out test set.

## Contents
1. Setup and loading
2. Dataset overview and summary table
3. Data-quality checks

## 1. Setup and loading

In [9]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make `src/` importable when the notebook runs from notebooks/
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.data_loader import (
    load_train, load_validation, split_columns,
    WAP_COLS, TARGET_COLS, META_COLS, MISSING_RSSI,
)

FIG_DIR = REPO_ROOT / "results" / "figures"
METRICS_DIR = REPO_ROOT / "results" / "metrics"
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

In [10]:
train = load_train()
val = load_validation()

X_tr, y_tr, meta_tr = split_columns(train)
X_va, y_va, meta_va = split_columns(val)

print("Train:     ", train.shape)
print("Validation:", val.shape)

Train:      (19937, 529)
Validation: (1111, 529)


In [11]:
# The last 9 (non-WAP) columns
train[TARGET_COLS + META_COLS].head()

,BUILDINGID,FLOOR,LONGITUDE,LATITUDE,SPACEID,RELATIVEPOSITION,USERID,PHONEID,TIMESTAMP
0,1,2,-7541.2643,4.864921e+06,106,2,2,23,1371713733
1,1,2,-7536.6212,4.864934e+06,106,2,2,23,1371713691
2,1,2,-7519.1524,4.864950e+06,103,2,2,23,1371714095
3,1,2,-7524.5704,4.864934e+06,102,2,2,23,1371713807
4,0,0,-7632.1436,4.864982e+06,122,2,11,13,1369909710


## 2. Dataset overview

The 529 columns split into three groups:

| Group | Columns | Use |
|---|---|---|
| Inputs | `WAP001`–`WAP520` | RSSI in dBm (−104 to 0), or **100 = not detected** |
| Targets | `BUILDINGID`, `FLOOR` (classification); `LONGITUDE`, `LATITUDE` (regression) | What we predict |
| Metadata | `SPACEID`, `RELATIVEPOSITION`, `USERID`, `PHONEID`, `TIMESTAMP` | Robustness and leakage analysis only |

In [12]:
def n_detected(X):
    """Number of access points actually detected in each scan."""
    return (X != MISSING_RSSI).sum(axis=1)

def summarise(df, X, name):
    det = n_detected(X)
    ts = pd.to_datetime(df["TIMESTAMP"], unit="s")
    return {
        "split": name,
        "rows": len(df),
        "columns": df.shape[1],
        "WAP columns": len(WAP_COLS),
        "buildings": df["BUILDINGID"].nunique(),
        "floors": df["FLOOR"].nunique(),
        "building-floor pairs": df.groupby(["BUILDINGID", "FLOOR"]).ngroups,
        "users": df["USERID"].nunique(),
        "phones": df["PHONEID"].nunique(),
        "first capture": ts.min().date(),
        "last capture": ts.max().date(),
        "% WAP cells = 100 (not detected)": round(100 * (X == MISSING_RSSI).values.mean(), 2),
        "mean APs detected per scan": round(det.mean(), 1),
        "median APs detected per scan": int(det.median()),
        "scans with 0 APs detected": int((det == 0).sum()),
        "exact duplicate rows": int(df.duplicated().sum()),
        "null values": int(df.isna().sum().sum()),
    }

summary = pd.DataFrame([summarise(train, X_tr, "train"),
                        summarise(val, X_va, "validation")]).set_index("split").T
summary.to_csv(METRICS_DIR / "eda_dataset_summary.csv")
summary

split,train,validation
rows,19937,1111
columns,529,529
WAP columns,520,520
buildings,3,3
floors,5,5
building-floor pairs,13,13
users,18,1
phones,16,11
first capture,2013-05-30,2013-09-19
last capture,2013-06-20,2013-10-08


**Reading the summary**

- The dataset has **no NaN values**, but that is misleading: absent access points are encoded as
  `100`, so the data is in fact extremely sparse. Most WAP cells are "not detected".
- A typical scan sees only a small fraction of the 520 access points. This sparsity is the core
  modelling challenge and the reason preprocessing (Member 3) must treat `100` explicitly.
- The two files were collected on different dates, so some distribution shift between them is expected.

## 3. Data-quality checks

### 3.1 RSSI values are within the documented range

In [13]:
def rssi_value_check(X, name):
    v = X.values.ravel()
    observed = v[v != MISSING_RSSI]
    return {
        "split": name,
        "min observed RSSI": int(observed.min()),
        "max observed RSSI": int(observed.max()),
        "values outside [-104, 0] (excl. 100)": int(((observed < -104) | (observed > 0)).sum()),
    }

pd.DataFrame([rssi_value_check(X_tr, "train"), rssi_value_check(X_va, "validation")]).set_index("split")

,min observed RSSI,max observed RSSI,"values outside [-104, 0] (excl. 100)"
split,,,
train,-104,0,0
validation,-102,-34,0


Every real measurement should lie between −104 and 0 dBm. Any non-zero count in the last column
indicates corrupted values that must be reported. Note that `100` is **not** a strong signal:
real RSSI is always ≤ 0, and values closer to 0 mean *stronger* signals.

### 3.2 Label validity

In [14]:
for name, df in [("train", train), ("validation", val)]:
    print(f"{name}: buildings = {sorted(df['BUILDINGID'].unique().tolist())}, floors = {sorted(df['FLOOR'].unique().tolist())}")

# Which floors exist in each building?
floors_per_building = (
    pd.concat([train[["BUILDINGID", "FLOOR"]].assign(split="train"),
               val[["BUILDINGID", "FLOOR"]].assign(split="validation")])
      .groupby(["split", "BUILDINGID"])["FLOOR"]
      .apply(lambda s: sorted(s.unique().tolist()))
      .unstack("split")
)
floors_per_building

train: buildings = [0, 1, 2], floors = [0, 1, 2, 3, 4]
validation: buildings = [0, 1, 2], floors = [0, 1, 2, 3, 4]


split,train,validation
BUILDINGID,,
0,"[0, 1, 2, 3]","[0, 1, 2, 3]"
1,"[0, 1, 2, 3]","[0, 1, 2, 3]"
2,"[0, 1, 2, 3, 4]","[0, 1, 2, 3, 4]"


Buildings should be 0–2 and floors 0–4. Floor 4 should appear only in building 2, the one
building with five floors. A floor label of 4 in buildings 0 or 1 would be invalid.

In [15]:
invalid = train[(train["FLOOR"] == 4) & (train["BUILDINGID"] != 2)]
print("Train rows with floor 4 outside building 2:", len(invalid))

# Coordinate ranges per building (sanity check that buildings are spatially separate)
train.groupby("BUILDINGID")[["LONGITUDE", "LATITUDE"]].agg(["min", "max"]).round(1)

Train rows with floor 4 outside building 2: 0


LONGITUDE           LATITUDE           
                 min     max        min        max
BUILDINGID                                        
0            -7691.3 -7587.0  4864897.5  4865016.7
1            -7578.5 -7404.5  4864810.0  4864959.5
2            -7415.2 -7300.8  4864745.7  4864861.8

### 3.3 Duplicates

In [16]:
dup_full = train.duplicated()
dup_fp_label = train.duplicated(subset=WAP_COLS + TARGET_COLS)
dup_fp_only = train.duplicated(subset=WAP_COLS)

print(f"Exact duplicate rows (all 529 columns):        {dup_full.sum():>5}")
print(f"Same fingerprint + same location:               {dup_fp_label.sum():>5}")
print(f"Same fingerprint, any location:                 {dup_fp_only.sum():>5}")
print(f"Validation exact duplicates:                    {val.duplicated().sum():>5}")

Exact duplicate rows (all 529 columns):          637
Same fingerprint + same location:                 754
Same fingerprint, any location:                   852
Validation exact duplicates:                        0


In [17]:
# Identical fingerprints labelled with DIFFERENT locations are ambiguous for any model.
# Empty scans (0 APs detected) are excluded here; they are covered in section 3.4.
has_signal = n_detected(X_tr) > 0
same_fp = train[has_signal & train.duplicated(subset=WAP_COLS, keep=False)]
conflicting = (
    same_fp.groupby(WAP_COLS)[["BUILDINGID", "FLOOR"]]
           .nunique()
           .max(axis=1)
)
print("Distinct fingerprints that map to more than one building/floor:",
      int((conflicting > 1).sum()))

Distinct fingerprints that map to more than one building/floor: 0


Exact duplicates usually arise from repeated scans at the same reference point. They inflate the
apparent dataset size and can leak information if one copy lands in a training fold and another in a
validation fold.

**Finding:** 637 exact duplicate rows (3.2% of train). A further 117 scans repeat a fingerprint and its
location but differ in metadata (e.g. timestamp). Importantly, **no fingerprint with signal maps to two
different building/floor labels**, so the duplicates are redundant rather than contradictory.
**The team should decide together whether to drop them** (Member 3 implements this; Member 4 must use
the same choice for cross-validation).

### 3.4 Scans with no detected access points

In [18]:
det_tr = n_detected(X_tr)
det_va = n_detected(X_va)

zero_tr = train[det_tr == 0]
print("Train scans with 0 APs:     ", len(zero_tr))
print("Validation scans with 0 APs:", int((det_va == 0).sum()))

if len(zero_tr):
    display(zero_tr.groupby(["BUILDINGID", "FLOOR"]).size().rename("zero-AP scans").to_frame())
    display(zero_tr.groupby(["USERID", "PHONEID"]).size().rename("zero-AP scans").to_frame())

Train scans with 0 APs:      76
Validation scans with 0 APs: 0


zero-AP scans
BUILDINGID FLOOR               
0          0                  1
1          3                 37
2          0                 36
           1                  1
           2                  1

,,zero-AP scans
USERID,PHONEID,
7,6,1
8,1,59
11,13,2
17,22,14


A scan with no detected access points carries **no Wi-Fi information**, so no model can locate it.

**Finding:** all 76 empty scans are in the training file, and **59 of them (78%) come from a single
user/phone (USERID 8, PHONEID 1)**, concentrated in building 1 floor 3 and building 2 floor 0.
This points to a faulty collection session, not a property of those locations. These rows should be
removed from training; the validation file has none.

### 3.5 Access points never detected

In [19]:
seen_tr = (X_tr != MISSING_RSSI).any(axis=0)
seen_va = (X_va != MISSING_RSSI).any(axis=0)

wap_coverage = pd.Series({
    "WAPs seen in train": int(seen_tr.sum()),
    "WAPs seen in validation": int(seen_va.sum()),
    "never seen in train (constant columns)": int((~seen_tr).sum()),
    "seen in validation but NOT in train": int((seen_va & ~seen_tr).sum()),
    "seen in train but NOT in validation": int((seen_tr & ~seen_va).sum()),
    "never seen anywhere": int((~seen_tr & ~seen_va).sum()),
}, name="count")
wap_coverage.to_frame()

,count
WAPs seen in train,465
WAPs seen in validation,367
never seen in train (constant columns),55
seen in validation but NOT in train,55
seen in train but NOT in validation,153
never seen anywhere,0


- **Never seen in train:** these columns are constant (always 100) in training data, so a model
  cannot learn anything from them. They can be dropped. The decision must be based on the
  **training file only**.
- **Seen in validation but not in train:** access points that appear at test time but were
  never available to learn from. This is one concrete form of train/validation shift and
  will be explored further in stage 4.

In [20]:
never_seen_train = [c for c in WAP_COLS if not seen_tr[c]]
print(f"{len(never_seen_train)} constant WAP columns in train, e.g. {never_seen_train[:8]}")

55 constant WAP columns in train, e.g. ['WAP003', 'WAP004', 'WAP092', 'WAP093', 'WAP094', 'WAP095', 'WAP152', 'WAP158']


### 3.6 Metadata placeholders in the validation file

In [21]:
for col in ["SPACEID", "RELATIVEPOSITION", "USERID"]:
    print(f"{col:<17} train unique: {train[col].nunique():>4}   validation unique values: {sorted(val[col].unique().tolist())}")
print(f"{'PHONEID':<17} phones in both files: {sorted(set(train['PHONEID']) & set(val['PHONEID']))}")

SPACEID           train unique:  123   validation unique values: [0]
RELATIVEPOSITION  train unique:    2   validation unique values: [0]
USERID            train unique:   18   validation unique values: [0]
PHONEID           phones in both files: [13, 14]


In the validation file, `SPACEID`, `RELATIVEPOSITION` and `USERID` are placeholders (all 0), so
they cannot be used as features. User-grouped robustness splits are possible **only within the
training file**.

**Finding:** the validation file uses 11 phones, but **only 2 of them (PHONEID 13 and 14) also appear in
training**. Different phones report different RSSI for the same location, so the validation set tests
generalisation to largely *unseen devices*. This, together with the 3-month gap between collections and
the access points that appear only in validation, means validation errors will be higher than
cross-validation errors within the training file. That is expected and is not a modelling bug.

## Stage 1 findings

| Check | Result | Action / owner |
|---|---|---|
| Missing values | No NaNs; absence encoded as `100` (96.5% of WAP cells in train, 96.8% in validation) | Member 3: impute consistently |
| Sparsity | Median 17 of 520 APs detected per scan in train, 15 in validation | Core modelling challenge |
| RSSI range | 0 values outside [−104, 0] dBm in either file | None needed |
| Labels | Buildings 0–2, floors 0–4, floor 4 only in building 2; 13 building–floor pairs in both files | None needed |
| Spatial separation | Coordinate ranges of the 3 buildings barely overlap | Building is well defined |
| Duplicates | 637 exact duplicate rows in train (3.2%); none conflicting; 0 in validation | Team decision: drop or keep |
| Zero-AP scans | 76 in train (59 from USERID 8 / PHONEID 1), 0 in validation | Remove from training |
| Constant WAPs | 55 never detected in train | Member 3: drop these columns |
| WAPs absent from validation | 153 seen in train but never in validation | Note: these features carry no signal at test time |
| Unseen-in-train WAPs | The same 55 appear only in validation | Distribution shift; cannot be learned |
| Devices | Only 2 of the 11 validation phones appear in train | Distribution shift; explains higher test error |
| Collection dates | Train 30 May – 20 Jun 2013; validation 19 Sep – 8 Oct 2013 | Temporal shift |
| Validation metadata | `USERID`, `SPACEID`, `RELATIVEPOSITION` are placeholders (0) | Grouped splits within train only |